In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [2]:
%config InlineBackend.figure_format = 'retina'

import random

from nsppk import NSPPK

from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator


In [3]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 250
min_num_nodes = 14
max_num_nodes = 16

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


dataset: zinc_250k
n_graphs: 250
node_range: [14, 16]


In [4]:
df = add(
    compose(name("cyc"), cycle()),
    compose(name("tree"), tree()),
)
decomposition_function = compose(intersection_edges(), df)
nbits = 14

neighbor_vectorizer = NSPPK(
    radius=1,
    distance=4,
    connector=1,
    nbits=14,
    parallel=True,
)

generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=neighbor_vectorizer,
    n_jobs=1,
    debug=True,
    debug_level=1,
)


In [5]:
%%time
generator.store(graphs, neighbor_vectorizer=neighbor_vectorizer)
print(f"stored_graphs = {len(generator.stored_graphs_)}")


stored_graphs = 250
CPU times: user 303 ms, sys: 561 ms, total: 864 ms
Wall time: 1.38 s


## Local Conditional Generation

Sample one stored ZINC molecule at random, retrieve its nearest stored neighbors,
fit `ConditionalAutoregressiveGenerator` on that local set, and generate from
the sampled molecule's interpretation graph.

In [6]:
%%time
n_neighbors = 7
n_samples = 7

samples = generator.sample(
    n_samples=n_samples,
    n_neighbors=n_neighbors,
    random_state=0,
    max_backtracks=12000,
    max_attempts_per_sample=48,
    max_total_attempts=256,
    progress_every_attempts=50,
    progress_every_seconds=10.0,
)

print(f"sampled_index = {generator.last_sampled_index_}")
print(f"neighbor_indices = {generator.last_neighbor_indices_}")
print(f"training_graphs = {len(generator.last_generation_training_graphs_)}")
print(f"generated = {len(samples)}")


[DEBUG] event=fit_dictionaries
[DEBUG]   bucket_keys=3
[DEBUG]   components=24
[DEBUG]   interpretation_pool=7
[DEBUG]   inv_freq_keys=5
[DEBUG]   inv_keys=5
[DEBUG]   skipped_missing_anchor_components=0
[DEBUG] event=fit_distributions_index
[DEBUG]   bucket_size=min/mean/max=(4/8.00/14)
[DEBUG]   inv_members=min/mean/max=(2/5.20/12)
[DEBUG] event=fit_distributions_components
[DEBUG]   anchors_per_port=min/mean/max=(1/1.00/1)
[DEBUG]   component_degree=min/mean/max=(1/1.42/2)
[DEBUG]   ports_per_component=min/mean/max=(1/1.42/2)
[DEBUG] event=fit_anchors
[DEBUG]   inv_multiplicity_hist={1: 5}
[DEBUG]   unique_anchor_types=2
[DEBUG] event=fit_anchor_hist
[DEBUG]   anchors_per_base_subgraph_hist={1: 14, 2: 10}
[DEBUG] event=fit_top_buckets
[DEBUG]   top_buckets=[{'key': (134332, 1), 'size': 14}, {'key': (462549, 2), 'size': 6}, {'key': (279703, 2), 'size': 4}]
[DEBUG] event=generate_indexes
[DEBUG]   bucket_keys=3
[DEBUG]   components=24
[DEBUG]   inv_freq_keys=5
[DEBUG]   inv_keys=5
[DE

NameError: name 'warnings' is not defined

In [ ]:
source_graph = generator.stored_graphs_[generator.last_sampled_index_]

print("Source molecule:")
display_graphs([source_graph], n_graphs_per_line=1)

print("Local fitting set:")
display_graphs(generator.last_generation_training_graphs_, n_graphs_per_line=4)

print("Generated molecules:")
display_graphs(samples, n_graphs_per_line=7)


---